  ## Proyecto Final: Mesa de Ayuda IA para Recursos Humanos

# 1. Introducción

Se desarrolló un prototipo de mesa de ayuda inteligente para Recursos Humanos de Patito S.A., utilizando **LangChain, Google Gemini y RAG**.

El sistema integra **tres agentes RAG** especializados en Beneficios y Compensaciones, Políticas Internas y Reclutamiento y Onboarding. Además, cuenta con un **agente de acción** para registrar solicitudes de Recursos Humanos, como vacaciones e inscripción de dependientes, mediante el archivo `registro_solicitudes_rrhh.txt`.

Un **orquestador** coordina los agentes según la intención de la consulta del usuario y genera la respuesta final.

## Arquitectura conceptual

                         USUARIO
                            │
                            ▼
                      ORQUESTADOR
                            │
              ┌─────────────┼─────────────┐
              ▼             ▼             ▼
         BENEFICIOS      POLÍTICAS    RECLUTAMIENTO
            RAG             RAG           RAG
              │             │             │
              └─────────────┼─────────────┘
                            │
                            ▼
                    AGENTE DE ACCIÓN
                            │
                            ▼
                registrar_solicitud_rrhh()
                            │
                            ▼
              registro_solicitudes_rrhh.txt
                            │
                            ▼
                     RESPUESTA FINAL



## 2. Instalación de librerías

Se instalan las dependencias necesarias para desarrollar el sistema con LangChain, integrar Google Gemini, construir las bases de conocimiento RAG y utilizar ChromaDB como almacenamiento vectorial.

También se incluye `python-dotenv` para gestionar variables de entorno e `ipywidgets` para componentes interactivos en Jupyter.


In [1]:
# langchain: framework para construir agentes, RAG, tools y orquestación.
# langchain-google-genai: integración con Google Gemini y sus embeddings.
# langchain-community: componentes e integraciones adicionales de LangChain.
# langchain-chroma: integración entre LangChain y Chroma.
# chromadb: almacenamiento vectorial para las bases de conocimiento RAG.
# python-dotenv: carga segura de variables de entorno desde un archivo .env.
# ipywidgets: biblioteca para crear interfaces interactivas y widgets en Jupyter.

%pip install -U langchain langchain-google-genai langchain-community langchain-chroma chromadb python-dotenv ipywidgets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 3. Importación de librerías

Se importan las herramientas necesarias para desarrollar el proyecto. Estas permiten gestionar variables de entorno, conectar los modelos de Google Gemini, cargar y dividir los documentos, y utilizar Chroma como almacenamiento vectorial para las bases de conocimiento de los agentes RAG.


In [2]:
# Importa os para trabajar con variables de entorno
# y acceder de forma segura a la GOOGLE_API_KEY.
import os

# Permite cargar las variables de entorno almacenadas
# en el archivo .env.
from dotenv import load_dotenv

# Modelo de lenguaje de Google Gemini que utilizaremos
# para generar las respuestas de nuestros agentes y orquestador.
from langchain_google_genai import ChatGoogleGenerativeAI

# Modelo de embeddings de Google Gemini que utilizaremos
# para convertir los documentos en vectores.
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Permite cargar los documentos de texto (.txt)
# que utilizaremos como bases de conocimiento.
from langchain_community.document_loaders import TextLoader

# Permite dividir los documentos en fragmentos (chunks)
# antes de generar los embeddings.
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Permite utilizar Chroma como vector store
# para almacenar y buscar los embeddings.
from langchain_chroma import Chroma
print("Librerías importadas correctamente.")

C:\Users\USER\AppData\Local\Temp\ipykernel_15728\4219190816.py:19: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Librerías importadas correctamente.


## 4. Configuración de la API de Google Gemini

Se crea un archivo `.env` dentro de la carpeta del proyecto para almacenar de forma segura la clave de acceso a Google Gemini. La clave se solicita mediante `getpass` para evitar mostrarla directamente en pantalla.

Luego, se carga la variable `GOOGLE_API_KEY` desde el archivo `.env` y se verifica que esté disponible antes de continuar con la ejecución del proyecto.


 4A — Creación del archivo .env

In [3]:
from pathlib import Path
from getpass import getpass

# 1. Busca específicamente la carpeta MI PROYECTO dentro de tu usuario
ruta_carpeta_proyecto = Path.home() / "MI PROYECTO"

# Asegurar de que la carpeta exista por si acaso
ruta_carpeta_proyecto.mkdir(parents=True, exist_ok=True)

# 2. Define la ruta exacta del .env dentro de esa carpeta
ruta_env_final = ruta_carpeta_proyecto / ".env"

# Solicita la clave de forma segura
GOOGLE_API_KEY = getpass("Ingresa tu GOOGLE_API_KEY: ")

# 3. Escribe el archivo directamente en MI PROYECTO
with open(ruta_env_final, "w") as archivo:
    archivo.write(f"GOOGLE_API_KEY={GOOGLE_API_KEY}\n")

print(f"¡Listo! Archivo .env creado correctamente en: {ruta_env_final}")

Ingresa tu GOOGLE_API_KEY:  ········


¡Listo! Archivo .env creado correctamente en: C:\Users\USER\MI PROYECTO\.env


In [4]:
# Carga las variables de entorno definidas en el archivo .env.
# En este archivo tendremos almacenada nuestra GOOGLE_API_KEY.
load_dotenv()

# Obtiene la clave API de Google Gemini desde la variable de entorno.
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# Verifica que la clave API haya sido encontrada correctamente.
# Si no existe, se detiene la ejecución y muestra un mensaje de error.
if not GOOGLE_API_KEY:
    raise ValueError(
        "No se encontró la GOOGLE_API_KEY. "
        "Verifica que esté configurada en el archivo .env."
    )

# Configura la clave API para que las herramientas de Google Gemini
# puedan utilizarla durante la ejecución del proyecto.
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

print("GOOGLE_API_KEY configurada correctamente.")

GOOGLE_API_KEY configurada correctamente.



## 5. Configuración de modelos de IA

Se definen los modelos de **Google Gemini** que utilizará el proyecto. El modelo LLM será utilizado por los agentes y el orquestador para interpretar consultas y generar respuestas. También se define el modelo de embeddings, que posteriormente permitirá transformar los documentos y consultas en representaciones vectoriales para las bases de conocimiento RAG.


In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Modelo de lenguaje (LLM) utilizado por los agentes y el orquestador.
# Se encargará de interpretar las consultas y generar las respuestas.
MODELO_LLM = "gemini-3.1-flash-lite"

# Modelo utilizado para generar embeddings.
# Convierte los documentos y consultas en representaciones vectoriales
# que serán almacenadas y consultadas mediante Chroma.
MODELO_EMBEDDING = "gemini-embedding-2-preview"

# Creamos el objeto LLM listo para ser usado por los agentes y el orquestador
llm = ChatGoogleGenerativeAI(model=MODELO_LLM, temperature=0)

# Muestra los modelos configurados para comprobar que los valores
# fueron definidos correctamente.
print("Modelo LLM:", MODELO_LLM)
print("Modelo de Embeddings:", MODELO_EMBEDDING)
print("Objeto LLM inicializado correctamente.")

Modelo LLM: gemini-3.1-flash-lite
Modelo de Embeddings: gemini-embedding-2-preview
Objeto LLM inicializado correctamente.


## 5.1 Configuración central de rutas del proyecto

Se centralizan las rutas de los documentos, los vector stores y el archivo de registro de solicitudes. La ruta principal se obtiene mediante `Path.cwd()` y, a partir de ella, se construyen las rutas utilizadas por los diferentes agentes del proyecto.

El archivo `registro_solicitudes_rrhh.txt` se utiliza para almacenar las solicitudes registradas por el agente de acción y se crea automáticamente cuando se realiza el primer registro.

Esta configuración facilita la organización y portabilidad del proyecto, evitando depender de rutas absolutas específicas de un computador.


In [6]:
from pathlib import Path
import os

# ============================================================
# CONFIGURACIÓN CENTRAL DE RUTAS DEL PROYECTO
# ============================================================

# Carpeta principal del proyecto
RUTA_PROYECTO = Path.cwd() 

print("Ruta principal del proyecto:")
print(RUTA_PROYECTO)

# ------------------------------------------------------------
# CARPETA DE DOCUMENTOS
# ------------------------------------------------------------

RUTA_DOCUMENTOS = RUTA_PROYECTO / "Documentos"

RUTA_DOCUMENTO_BENEFICIOS = (
    RUTA_DOCUMENTOS / "01_Beneficios_Compensaciones.txt"
)

RUTA_DOCUMENTO_REGLAMENTO = (
    RUTA_DOCUMENTOS / "02_Reglamento_Interno.txt"
)

RUTA_DOCUMENTO_RECLUTAMIENTO = (
    RUTA_DOCUMENTOS / "03_Reclutamiento_Onboarding.txt"
)

# ------------------------------------------------------------
# CARPETA DE VECTOR STORES
# ------------------------------------------------------------

RUTA_VECTORSTORES = RUTA_PROYECTO / "Vectorstores"

RUTA_VECTORSTORE_BENEFICIOS = (
    RUTA_VECTORSTORES / "beneficios_compensaciones"
)

RUTA_VECTORSTORE_REGLAMENTO = (
    RUTA_VECTORSTORES / "reglamento_interno"
)

RUTA_VECTORSTORE_RECLUTAMIENTO = (
    RUTA_VECTORSTORES / "reclutamiento_onboarding"
)

# ------------------------------------------------------------
# ARCHIVO DE REGISTRO DE SOLICITUDES
# ------------------------------------------------------------

REGISTRO_PATH = (
    RUTA_PROYECTO / "registro_solicitudes_rrhh.txt"
)

# ============================================================
# VERIFICACIÓN DE RUTAS
# ============================================================

print("\nDocumentos:")

print(
    "Beneficios:",
    RUTA_DOCUMENTO_BENEFICIOS,
    "->",
    RUTA_DOCUMENTO_BENEFICIOS.exists()
)

print(
    "Reglamento:",
    RUTA_DOCUMENTO_REGLAMENTO,
    "->",
    RUTA_DOCUMENTO_REGLAMENTO.exists()
)

print(
    "Reclutamiento:",
    RUTA_DOCUMENTO_RECLUTAMIENTO,
    "->",
    RUTA_DOCUMENTO_RECLUTAMIENTO.exists()
)

print("\nArchivo de solicitudes:")

print(
    "Registro:",
    REGISTRO_PATH,
    "->",
    REGISTRO_PATH.exists()
)

Ruta principal del proyecto:
C:\Users\USER\MI PROYECTO

Documentos:
Beneficios: C:\Users\USER\MI PROYECTO\Documentos\01_Beneficios_Compensaciones.txt -> True
Reglamento: C:\Users\USER\MI PROYECTO\Documentos\02_Reglamento_Interno.txt -> True
Reclutamiento: C:\Users\USER\MI PROYECTO\Documentos\03_Reclutamiento_Onboarding.txt -> True

Archivo de solicitudes:
Registro: C:\Users\USER\MI PROYECTO\registro_solicitudes_rrhh.txt -> False


## 6. Agente 1 — Beneficios y Compensaciones

Se desarrolla un agente RAG especializado en Beneficios y Compensaciones de Patito S.A.

El agente utiliza `01_Beneficios_Compensaciones.txt` como base de conocimiento. Se realiza la carga del documento, generación de embeddings, creación del vector store, configuración del retriever y construcción del agente RAG.

Finalmente, se realizan pruebas para comprobar que las respuestas se basen únicamente en la información del documento.


## 6.1 Carga de la base de conocimiento: Beneficios y Compensaciones

Se carga el archivo `01_Beneficios_Compensaciones.txt`, que contiene la información utilizada por el agente RAG de Beneficios y Compensaciones. El documento se prepara para las siguientes etapas de procesamiento y creación de la base vectorial.


In [32]:
# ============================================================
# CARGA DEL DOCUMENTO DE BENEFICIOS Y COMPENSACIONES
# ============================================================

# Carga el documento de Beneficios y Compensaciones.
# La ruta se obtiene desde la configuración central del proyecto.

loader_beneficios = TextLoader(
    str(RUTA_DOCUMENTO_BENEFICIOS),
    encoding="utf-8"
)

# Lee el contenido del documento y lo convierte
# en documentos que podremos procesar posteriormente.
documentos_beneficios = loader_beneficios.load()

# Muestra un mensaje para confirmar que el documento fue cargado.
print("Documento de Beneficios y Compensaciones cargado correctamente.")

Documento de Beneficios y Compensaciones cargado correctamente.


## 6.2. División del documento en fragmentos (Chunking)

Se divide el documento de Beneficios y Compensaciones en fragmentos pequeños llamados chunks. Esto facilita la búsqueda semántica y permite que el sistema RAG recupere únicamente la información relevante para responder cada consulta.


In [33]:
import re

# ============================================================
# PROCESAMIENTO DEL DOCUMENTO DE BENEFICIOS Y COMPENSACIONES
# ============================================================

# 1. Leemos el contenido completo del archivo de texto.
# La ruta se obtiene desde la configuración central del proyecto.
with open(RUTA_DOCUMENTO_BENEFICIOS, "r", encoding="utf-8") as f:
    texto_beneficios = f.read()


# 2. Función de partición exacta para los bloques principales
#    (1., 2., 3., 4.)
def chunkear_beneficios(texto):

    # Busca los números principales seguidos de punto y un espacio
    # al inicio de la línea (ej: "1. ", "2. ")
    cabeceras = list(
        re.finditer(
            r"^\d+\.\s+[A-ZÁÉÍÓÚÑ\s]+$",
            texto,
            flags=re.MULTILINE
        )
    )

    chunks = []

    for i, m in enumerate(cabeceras):

        ini = m.start()

        fin = (
            cabeceras[i + 1].start()
            if i + 1 < len(cabeceras)
            else len(texto)
        )

        parte = texto[ini:fin].strip()

        if parte:
            chunks.append(parte)

    return chunks


lista_textos_chunks = chunkear_beneficios(texto_beneficios)


# 3. Validación y estadísticas

if len(lista_textos_chunks) > 0:

    print(f"Total de chunks: {len(lista_textos_chunks)}")

    print(
        f"Tamaño promedio: "
        f"{sum(len(c) for c in lista_textos_chunks) // len(lista_textos_chunks)} "
        f"caracteres"
    )

    print(
        f"Tamaño min/max: "
        f"{min(len(c) for c in lista_textos_chunks)} / "
        f"{max(len(c) for c in lista_textos_chunks)} caracteres\n"
    )

    print("Inicio de cada chunk:")
    print("-" * 60)

    for i, c in enumerate(lista_textos_chunks):

        primeros = c[:55].replace("\n", " ")

        print(
            f"  ch_{i:02d}: {primeros}..."
        )

else:

    print(
        "⚠️ ADVERTENCIA: "
        "No se detectaron los bloques principales."
    )


# 4. Conversión a formato LangChain para Chroma

from langchain_core.documents import Document

chunks_beneficios = [
    Document(page_content=texto)
    for texto in lista_textos_chunks
]

Total de chunks: 4
Tamaño promedio: 280 caracteres
Tamaño min/max: 113 / 749 caracteres

Inicio de cada chunk:
------------------------------------------------------------
  ch_00: 1. SEGURO MÉDICO CORPORATIVO 1.1 Cobertura: consultas m...
  ch_01: 2. BONOS - Bono por desempeño anual según evaluación. -...
  ch_02: 3. OTROS BENEFICIOS - Día libre de cumpleaños. - Capaci...
  ch_03: 4. COMPENSACIÓN La estructura salarial considera el rol...


## 6.3. Generación de embeddings y creación del Vector Store

En esta sección se generan los embeddings del documento de Beneficios y Compensaciones utilizando Google Gemini. Los embeddings permiten representar los fragmentos del documento como vectores para realizar búsquedas semánticas. Estos vectores se almacenan en un Vector Store independiente de Chroma, que será utilizado posteriormente por el Agente RAG de Beneficios y Compensaciones.




In [29]:
import os

# ============================================================
# GENERACIÓN DE EMBEDDINGS Y CREACIÓN DEL VECTOR STORE
# ============================================================

# La ruta del Vector Store se obtiene desde la configuración
# central de rutas del proyecto:
#
# RUTA_VECTORSTORE_BENEFICIOS
#
# Esta ruta corresponde a:
# MI PROYECTO/Vectorstores/beneficios_compensaciones


# ============================================================
# 1. CREAR LA CARPETA DEL VECTOR STORE SI NO EXISTE
# ============================================================

# Crea la carpeta del Vector Store y todas las carpetas
# necesarias que no existan.
#
# exist_ok=True evita que aparezca un error si la carpeta
# ya existe.

os.makedirs(
    RUTA_VECTORSTORE_BENEFICIOS,
    exist_ok=True
)


# ============================================================
# 2. CREAR EL MODELO DE EMBEDDINGS
# ============================================================

# Crea el modelo de embeddings de Google Gemini.
# Este modelo convierte cada chunk del documento en una
# representación vectorial que puede ser almacenada
# y consultada posteriormente.

embeddings = GoogleGenerativeAIEmbeddings(
    model=MODELO_EMBEDDING
)


# ============================================================
# 3. CREAR EL VECTOR STORE DE CHROMA
# ============================================================

# Convierte los chunks del documento en embeddings y los
# almacena en una base vectorial de Chroma.
#
# Este Vector Store pertenece exclusivamente al agente
# de Beneficios y Compensaciones.

vectorstore_beneficios = Chroma.from_documents(
    documents=chunks_beneficios,
    embedding=embeddings,
    persist_directory=str(RUTA_VECTORSTORE_BENEFICIOS)
)


# ============================================================
# 4. VERIFICAR QUE EL PROCESO TERMINÓ CORRECTAMENTE
# ============================================================

print(
    "Vector Store de Beneficios y Compensaciones "
    "creado correctamente."
)

print("\nRuta del Vector Store:")
print(RUTA_VECTORSTORE_BENEFICIOS)


# Muestra los archivos creados por Chroma.
print("\nContenido generado por Chroma:")

for archivo in os.listdir(RUTA_VECTORSTORE_BENEFICIOS):
    print("-", archivo)

Vector Store de Beneficios y Compensaciones creado correctamente.

Ruta del Vector Store:
C:\Users\USER\MI PROYECTO\Vectorstores\beneficios_compensaciones

Contenido generado por Chroma:
- 15cb2097-738d-48d4-8f2d-e16175108bd1
- chroma.sqlite3


## 6.4. Creación del Retriever del Agente de Beneficios y Compensaciones

En esta sección se configura el retriever del Agente de Beneficios y Compensaciones. El retriever realiza búsquedas semánticas dentro del Vector Store creado previamente y recupera los fragmentos más relevantes para cada consulta del usuario.

Estos fragmentos serán utilizados posteriormente por el modelo de lenguaje para generar respuestas basadas exclusivamente en la base documental del agente.



In [10]:
# ============================================================
# 1. CREACIÓN DEL RETRIEVER
# ============================================================

# Convierte el Vector Store en un Retriever.
# El Retriever permite realizar búsquedas semánticas
# dentro de la base de conocimiento del agente.

retriever_beneficios = vectorstore_beneficios.as_retriever(
    search_kwargs={"k": 5}
)


# ============================================================
# 2. VERIFICACIÓN DEL RETRIEVER
# ============================================================

print(
    "Retriever de Beneficios y Compensaciones "
    "creado correctamente."
)

print(
    "Cantidad de documentos recuperados por consulta: 5"
)

Retriever de Beneficios y Compensaciones creado correctamente.
Cantidad de documentos recuperados por consulta: 5


## 6.5. Creación del Prompt del Agente RAG de Beneficios y Compensaciones

En esta sección se define el prompt que establece el comportamiento del Agente RAG de Beneficios y Compensaciones.

El agente debe responder utilizando únicamente la información recuperada desde su propia base de conocimiento. No debe inventar información ni utilizar conocimientos externos para responder consultas relacionadas con la empresa.

Cuando la información necesaria no se encuentre disponible en los documentos recuperados, el agente debe indicar explícitamente:

**"No encontré información suficiente en la base documental proporcionada."**

De esta manera, el agente cumple con las reglas y restricciones establecidas para el proyecto.


In [11]:
# ============================================================
# 1. PROMPT DEL AGENTE DE BENEFICIOS Y COMPENSACIONES
# ============================================================

# Define las instrucciones que seguirá el agente RAG.
# El contexto será proporcionado posteriormente por el Retriever.

PROMPT_BENEFICIOS = """
Eres el Agente de Beneficios y Compensaciones de PATITO S.A.

Tu función es responder preguntas relacionadas exclusivamente
con los beneficios y compensaciones de los colaboradores de
PATITO S.A.

Debes utilizar únicamente la información proporcionada en el
contexto documental recuperado desde la base de conocimiento
del Agente de Beneficios y Compensaciones.

REGLAS IMPORTANTES:

1. No inventes información que no aparezca en el contexto
   documental proporcionado.

2. No utilices conocimientos externos para completar una
   respuesta.

3. No mezcles información de otras bases de conocimiento
   pertenecientes a otros agentes.

4. Si la información necesaria para responder la pregunta
   no se encuentra en el contexto proporcionado, responde
   exactamente:

   "No encontré información suficiente en la base documental proporcionada."

5. Responde de manera clara, precisa y profesional.

6. Si la pregunta contiene información que sí aparece en el
   contexto documental, responde basándote exclusivamente
   en dicha información.

CONTEXTO DOCUMENTAL:
{context}

PREGUNTA DEL USUARIO:
{question}

RESPUESTA:
"""


# ============================================================
# 2. VERIFICACIÓN DEL PROMPT
# ============================================================

print("Prompt del Agente de Beneficios y Compensaciones creado correctamente.")
print("El prompt incluye las reglas de uso exclusivo de la base documental.")

Prompt del Agente de Beneficios y Compensaciones creado correctamente.
El prompt incluye las reglas de uso exclusivo de la base documental.


## 6.6. Creación del Agente RAG de Beneficios y Compensaciones

En esta sección se integran el modelo de lenguaje de Google Gemini, el Retriever y el prompt definido anteriormente para construir el primer agente RAG del proyecto.

El agente recibe una pregunta del usuario, utiliza el Retriever para localizar los fragmentos más relevantes de la base documental de Beneficios y Compensaciones y proporciona estos fragmentos al modelo de lenguaje como contexto.

De esta manera, el agente genera respuestas basadas exclusivamente en la información recuperada desde su propio Vector Store, evitando mezclar información con las bases de conocimiento de otros agentes.



In [12]:
# ============================================================
# 1. CREACIÓN DEL MODELO DE LENGUAJE
# ============================================================

# Inicializa el modelo de lenguaje de Google Gemini.
# Este modelo será utilizado para generar las respuestas
# del Agente de Beneficios y Compensaciones.

llm_beneficios = ChatGoogleGenerativeAI(
    model=MODELO_LLM,
    temperature=0
)


# ============================================================
# 2. FUNCIÓN DEL AGENTE RAG
# ============================================================

def agente_beneficios(pregunta):
    """
    Recibe una pregunta del usuario, busca información
    relevante en el Vector Store de Beneficios y
    Compensaciones y genera una respuesta utilizando Gemini.
    """

    # --------------------------------------------------------
    # Recuperar documentos relevantes
    # --------------------------------------------------------

    documentos_relevantes = retriever_beneficios.invoke(
        pregunta
    )

    # --------------------------------------------------------
    # Verificar si se encontró información
    # --------------------------------------------------------

    if not documentos_relevantes:
        return "No encontré información suficiente en la base documental proporcionada."

    # --------------------------------------------------------
    # Unir el contenido de los documentos recuperados
    # --------------------------------------------------------

    contexto = "\n\n".join(
        documento.page_content
        for documento in documentos_relevantes
    )

    # --------------------------------------------------------
    # Construir el prompt con el contexto recuperado
    # --------------------------------------------------------

    prompt_final = PROMPT_BENEFICIOS.format(
        context=contexto,
        question=pregunta
    )

    # --------------------------------------------------------
    # Enviar el prompt al modelo Gemini
    # --------------------------------------------------------

    respuesta = llm_beneficios.invoke(
        prompt_final
    )

    # --------------------------------------------------------
    # Retornar únicamente el contenido de la respuesta
    # --------------------------------------------------------

    return respuesta.content


# ============================================================
# 3. VERIFICACIÓN DEL AGENTE
# ============================================================

print("Agente RAG de Beneficios y Compensaciones creado correctamente.")

Agente RAG de Beneficios y Compensaciones creado correctamente.


# 7. Agente 2 — Reglamento Interno

En esta sección se construye el segundo agente RAG del proyecto, especializado en el Reglamento Interno de PATITO S.A.

Este agente contará con su propia base documental, sus propios fragmentos de texto, sus propios embeddings y un Vector Store independiente del Agente de Beneficios y Compensaciones.

El agente utilizará únicamente la información disponible en su propia base de conocimiento y no mezclará información con otros agentes. Si una consulta no puede ser respondida con la información recuperada desde su base documental, deberá indicar:

**"No encontré información suficiente en la base documental proporcionada."**



## 7.1. Carga del documento de conocimiento

En esta sección se carga el documento `02_Reglamento_Interno.txt`, que contiene la información utilizada como base de conocimiento para el Agente de Reglamento Interno.

Este documento será procesado posteriormente para dividir su contenido en fragmentos y construir su propia base vectorial.


In [13]:
# ============================================================
# 1. CARGA DEL DOCUMENTO DE REGLAMENTO INTERNO
# ============================================================

# Carga el documento que contiene la información
# del Reglamento Interno de PATITO S.A.
#
# La ruta se obtiene desde la configuración central
# del proyecto.

loader_reglamento = TextLoader(
    str(RUTA_DOCUMENTO_REGLAMENTO),
    encoding="utf-8"
)


# Lee el contenido del documento.
documentos_reglamento = loader_reglamento.load()


# Verificación de la carga.
print(
    "Documento de Reglamento Interno "
    "cargado correctamente."
)

print(
    "Cantidad de documentos cargados:",
    len(documentos_reglamento)
)

Documento de Reglamento Interno cargado correctamente.
Cantidad de documentos cargados: 1


## 7.2. División del documento en fragmentos (Chunking)

Se divide el documento de Reglamento Interno en fragmentos pequeños llamados chunks. Esto facilita la búsqueda semántica y permite que el sistema RAG recupere únicamente la información relevante para responder cada consulta.


In [14]:
import re

# ============================================================
# DIVISIÓN DEL REGLAMENTO INTERNO EN CHUNKS
# ============================================================

# 1. Utilizamos la ruta centralizada del proyecto.
# La variable RUTA_DOCUMENTO_REGLAMENTO fue definida
# previamente en la configuración central de rutas.

ruta_archivo_reglamento = RUTA_DOCUMENTO_REGLAMENTO

# Verificamos que el documento exista antes de leerlo.
if not ruta_archivo_reglamento.exists():
    raise FileNotFoundError(
        f"No se encontró el documento de Reglamento Interno en:\n"
        f"{ruta_archivo_reglamento}"
    )

# 2. Leemos el contenido completo del archivo de Reglamento.
with open(
    ruta_archivo_reglamento,
    "r",
    encoding="utf-8"
) as f:
    texto_reglamento = f.read()


# ============================================================
# 3. FUNCIÓN PARA DIVIDIR EL REGLAMENTO EN CHUNKS
# ============================================================

def chunkear_reglamento(texto):

    # Busca encabezados principales con el formato:
    # 1. NOMBRE DE LA SECCIÓN
    # 2. NOMBRE DE LA SECCIÓN
    # 3. NOMBRE DE LA SECCIÓN

    cabeceras = list(
        re.finditer(
            r"^\d+\.\s+[A-ZÁÉÍÓÚÑ\s]+$",
            texto,
            flags=re.MULTILINE
        )
    )

    chunks = []

    # Recorremos cada encabezado encontrado.
    for i, m in enumerate(cabeceras):

        # Posición inicial del bloque actual.
        ini = m.start()

        # Posición final del bloque.
        if i + 1 < len(cabeceras):
            fin = cabeceras[i + 1].start()
        else:
            fin = len(texto)

        # Extraemos el bloque completo.
        parte = texto[ini:fin].strip()

        # Evitamos agregar bloques vacíos.
        if parte:
            chunks.append(parte)

    return chunks


# ============================================================
# 4. EJECUTAR LA DIVISIÓN DEL DOCUMENTO
# ============================================================

lista_textos_chunks_reglamento = chunkear_reglamento(
    texto_reglamento
)


# ============================================================
# 5. VALIDACIÓN Y ESTADÍSTICAS
# ============================================================

if len(lista_textos_chunks_reglamento) > 0:

    total_chunks = len(
        lista_textos_chunks_reglamento
    )

    tamano_promedio = (
        sum(
            len(c)
            for c in lista_textos_chunks_reglamento
        )
        // total_chunks
    )

    tamano_minimo = min(
        len(c)
        for c in lista_textos_chunks_reglamento
    )

    tamano_maximo = max(
        len(c)
        for c in lista_textos_chunks_reglamento
    )

    print(
        "Documento utilizado:"
    )
    print(
        ruta_archivo_reglamento
    )

    print(
        f"\nTotal de chunks: {total_chunks}"
    )

    print(
        f"Tamaño promedio: "
        f"{tamano_promedio} caracteres"
    )

    print(
        f"Tamaño min/max: "
        f"{tamano_minimo} / "
        f"{tamano_maximo} caracteres\n"
    )

    print(
        "Inicio de cada chunk:"
    )

    print("-" * 60)

    for i, c in enumerate(
        lista_textos_chunks_reglamento
    ):

        primeros = c[:55].replace(
            "\n",
            " "
        )

        print(
            f"  ch_{i:02d}: "
            f"{primeros}..."
        )

else:

    print(
        "⚠️ ADVERTENCIA: "
        "No se detectaron los bloques principales."
    )


# ============================================================
# 6. CONVERSIÓN A DOCUMENTOS LANGCHAIN
# ============================================================

from langchain_core.documents import Document

chunks_reglamento = [
    Document(page_content=texto)
    for texto in lista_textos_chunks_reglamento
]


print(
    "\nChunks de Reglamento Interno "
    "convertidos correctamente a documentos LangChain."
)

Documento utilizado:
C:\Users\USER\MI PROYECTO\Documentos\02_Reglamento_Interno.txt

Total de chunks: 5
Tamaño promedio: 250 caracteres
Tamaño min/max: 150 / 389 caracteres

Inicio de cada chunk:
------------------------------------------------------------
  ch_00: 1. JORNADA LABORAL Jornada de 40 horas semanales. Horar...
  ch_01: 2. VACACIONES - Cada colaborador tiene derecho a 15 día...
  ch_02: 3. PERMISOS - Permisos remunerados: por matrimonio, nac...
  ch_03: 4. CÓDIGO DE CONDUCTA Respeto, no discriminación, ambie...
  ch_04: 5. FALTAS Y SANCIONES Las faltas se clasifican en leves...

Chunks de Reglamento Interno convertidos correctamente a documentos LangChain.


## 7.3. Generación de embeddings y creación del Vector Store

En esta sección se generan los embeddings del documento de Reglamento Interno utilizando Google Gemini. Los embeddings permiten representar los fragmentos del documento como vectores para realizar búsquedas semánticas.

Estos vectores se almacenan en un Vector Store independiente de Chroma, que será utilizado posteriormente por el Agente RAG de Reglamento Interno.


In [15]:
import os

# ============================================================
# GENERACIÓN DE EMBEDDINGS Y CREACIÓN DEL VECTOR STORE
# ============================================================

# La ruta del Vector Store se obtiene desde la configuración
# central de rutas del proyecto:
#
# RUTA_VECTORSTORE_REGLAMENTO
#
# Esta ruta corresponde a:
# MI PROYECTO/Vectorstores/reglamento_interno


# ============================================================
# 1. CREAR LA CARPETA DEL VECTOR STORE SI NO EXISTE
# ============================================================

# Crea la carpeta del Vector Store y todas las carpetas
# necesarias que no existan.
#
# exist_ok=True evita que aparezca un error si la carpeta
# ya existe.
# Como trabajamos con pathlib, utilizamos .mkdir()

RUTA_VECTORSTORE_REGLAMENTO.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. CREAR EL MODELO DE EMBEDDINGS
# ============================================================

# Crea el modelo de embeddings de Google Gemini.
# Este modelo convierte cada chunk del documento en una
# representación vectorial que puede ser almacenada
# y consultada posteriormente.

embeddings = GoogleGenerativeAIEmbeddings(
    model=MODELO_EMBEDDING
)


# ============================================================
# 3. CREAR EL VECTOR STORE DE CHROMA
# ============================================================

# Convierte los chunks del documento en embeddings y los
# almacena en una base vectorial de Chroma.
#
# Este Vector Store pertenece exclusivamente al agente
# de Reglamento Interno.

vectorstore_reglamento = Chroma.from_documents(
    documents=chunks_reglamento,
    embedding=embeddings,
    persist_directory=str(RUTA_VECTORSTORE_REGLAMENTO)
)


# ============================================================
# 4. VERIFICAR QUE EL PROCESO TERMINÓ CORRECTAMENTE
# ============================================================

print(
    "Vector Store de Reglamento Interno "
    "creado correctamente."
)

print("\nRuta del Vector Store:")
print(RUTA_VECTORSTORE_REGLAMENTO)


# Muestra los archivos creados por Chroma.
print("\nContenido generado por Chroma:")

for archivo in os.listdir(RUTA_VECTORSTORE_REGLAMENTO):
    print("-", archivo)

Vector Store de Reglamento Interno creado correctamente.

Ruta del Vector Store:
C:\Users\USER\MI PROYECTO\Vectorstores\reglamento_interno

Contenido generado por Chroma:
- chroma.sqlite3
- f89cc595-ba90-4c00-92af-e29b2cbde9ee


## 7.4. Creación del Retriever del Agente de Reglamento Interno

En esta sección se configura el retriever del Agente de Reglamento Interno. El retriever realiza búsquedas semánticas dentro del Vector Store creado previamente y recupera los fragmentos más relevantes para cada consulta del usuario.

Estos fragmentos serán utilizados posteriormente por el modelo de lenguaje para generar respuestas basadas exclusivamente en la base documental del Reglamento Interno.


In [16]:
# ============================================================
#  CREACIÓN DEL RETRIEVER DEL AGENTE DE REGLAMENTO INTERNO
# ============================================================

# El Retriever permite realizar búsquedas semánticas
# dentro del Vector Store de Reglamento Interno.
#
# El parámetro k=3 indica que se recuperarán los 5 chunks
# más relevantes para cada consulta.

retriever_reglamento = vectorstore_reglamento.as_retriever(
    search_kwargs={"k": 5}
)

# Verificación del proceso.
print(
    "Retriever del Agente de Reglamento Interno "
    "creado correctamente."
)

Retriever del Agente de Reglamento Interno creado correctamente.


## 7.5. Creación del Prompt del Agente de Reglamento Interno

En esta sección se definen las instrucciones específicas del Agente de Reglamento Interno. El prompt establece que el agente debe responder utilizando únicamente la información recuperada desde su propio Vector Store.

El agente no debe inventar información ni utilizar conocimientos externos. Si no encuentra información suficiente para responder una consulta, deberá indicarlo explícitamente.


In [17]:
# ============================================================
  #CREACIÓN DEL PROMPT DEL AGENTE DE REGLAMENTO INTERNO
# ============================================================

# Prompt específico para el Agente de Reglamento Interno.
# Este agente solo puede utilizar la información contenida
# en su propia base documental.

prompt_reglamento = """
Eres el Agente de Reglamento Interno de PATITO S.A.

Tu función es responder preguntas relacionadas exclusivamente
con el Reglamento Interno de la empresa.

REGLAS IMPORTANTES:

1. Responde únicamente utilizando la información proporcionada
   en el contexto recuperado desde la base documental del
   Reglamento Interno.

2. No utilices información de otras bases de conocimiento,
   agentes o documentos.

3. No inventes, supongas ni completes información que no esté
   presente en el contexto proporcionado.

4. Si la información necesaria para responder la pregunta
   no aparece en el contexto, responde exactamente:

"No encontré información suficiente en la base documental proporcionada"

5. Mantén las respuestas claras, directas y relacionadas
   exclusivamente con el Reglamento Interno de PATITO S.A.

CONTEXTO DEL REGLAMENTO INTERNO:
{context}

PREGUNTA DEL USUARIO:
{question}

RESPUESTA:
"""

# Verificación del proceso.
print(
    "Prompt del Agente de Reglamento Interno "
    "creado correctamente."
)

print("\nEl prompt incluye las reglas de uso exclusivo "
      "de la base documental.")

Prompt del Agente de Reglamento Interno creado correctamente.

El prompt incluye las reglas de uso exclusivo de la base documental.


## 7.6. Creación del Agente RAG de Reglamento Interno

En esta sección se integran el modelo de lenguaje de Google Gemini, el Retriever y el prompt definido anteriormente para construir el Agente RAG de Reglamento Interno.

El agente utilizará el Retriever para recuperar los fragmentos más relevantes del Reglamento Interno y proporcionarlos al modelo de lenguaje como contexto. De esta manera, generará respuestas basadas exclusivamente en la información disponible en su propia base documental.


In [18]:
# ============================================================
#  CREACIÓN DEL AGENTE DE REGLAMENTO INTERNO
# ============================================================

def agente_reglamento(pregunta):

    # 1. Recuperar los documentos más relevantes
    documentos_relevantes = retriever_reglamento.invoke(
        pregunta
    )

    # 2. Verificar si se encontró información
    if not documentos_relevantes:
        return (
            "No encontré información suficiente en la "
            "base documental proporcionada."
        )

    # 3. Construir el contexto
    contexto = "\n\n".join(
        documento.page_content
        for documento in documentos_relevantes
    )

    # 4. Crear el prompt final
    prompt_final = prompt_reglamento.format(
        context=contexto,
        question=pregunta
    )

    # 5. Consultar el modelo Gemini
    try:

        respuesta = llm.invoke(
            prompt_final
        )

        return respuesta.content

    except Exception as e:

        # Mostrar el error real para facilitar
        # la identificación de problemas.
        print("ERROR REAL DE GEMINI:")
        print(type(e).__name__)
        print(e)

        return (
            "No fue posible procesar la consulta "
            "debido a un error del servicio."
        )


print(
    "Agente de Reglamento Interno creado correctamente."
)

print(
    "El agente utiliza exclusivamente el Retriever "
    "y el Vector Store de Reglamento Interno."
)

Agente de Reglamento Interno creado correctamente.
El agente utiliza exclusivamente el Retriever y el Vector Store de Reglamento Interno.


# 8. Agente 3 — Reclutamiento y Onboarding

En esta sección se construye el tercer agente RAG del proyecto, especializado en los procesos de Reclutamiento y Onboarding de PATITO S.A.

Este agente contará con su propia base documental, sus propios fragmentos de texto, sus propios embeddings y un Vector Store independiente de los agentes anteriores.

El agente utilizará únicamente la información disponible en su propia base de conocimiento y no mezclará información con otros agentes. Si una consulta no puede ser respondida con la información recuperada desde su base documental, deberá indicar:

**"No encontré información suficiente en la base documental proporcionada."**


## 8.1. Carga del documento de conocimiento

En esta sección se carga el documento `03_Reclutamiento_Onboarding.txt`, que contiene la información utilizada como base de conocimiento para el Agente de Reclutamiento y Onboarding.

Este documento será procesado posteriormente para dividir su contenido en fragmentos y construir su propia base vectorial.


In [19]:
# ============================================================
#   CARGA DEL DOCUMENTO DE RECLUTAMIENTO Y ONBOARDING
# ============================================================

# Carga el documento que contiene la información
# de Reclutamiento y Onboarding de PATITO S.A.
#
# La ruta se obtiene desde la configuración central
# del proyecto.

loader_reclutamiento = TextLoader(
    str(RUTA_DOCUMENTO_RECLUTAMIENTO),
    encoding="utf-8"
)

# Lee el contenido del documento.
documentos_reclutamiento = loader_reclutamiento.load()

# Verificación de la carga.
print("Documento de Reclutamiento y Onboarding cargado correctamente.")
print("Cantidad de documentos cargados:", len(documentos_reclutamiento))

Documento de Reclutamiento y Onboarding cargado correctamente.
Cantidad de documentos cargados: 1


## 8.2. División del documento en fragmentos (Chunking)

Se divide el documento de Reclutamiento y Onboarding en fragmentos pequeños llamados chunks. Esto facilita la búsqueda semántica y permite que el sistema RAG recupere únicamente la información relevante para responder cada consulta.


In [20]:
import re

# ============================================================
#   DIVISIÓN DE RECLUTAMIENTO Y ONBOARDING EN CHUNKS 
# ============================================================

# 1. Utilizamos la ruta centralizada del proyecto.
ruta_archivo_reclutamiento = RUTA_DOCUMENTO_RECLUTAMIENTO

# Verificamos que el documento exista antes de leerlo.
if not ruta_archivo_reclutamiento.exists():
    raise FileNotFoundError(
        f"No se encontró el documento de Reclutamiento y Onboarding en:\n"
        f"{ruta_archivo_reclutamiento}"
    )

with open(
    ruta_archivo_reclutamiento,
    "r",
    encoding="utf-8"
) as f:
    texto_reclutamiento = f.read()

def chunkear_reclutamiento(texto):
    # Patrón para detectar líneas que empiezan con un número y un punto
    cabeceras = list(
        re.finditer(
            r"^\d+\.\s+.*$",
            texto,
            flags=re.MULTILINE
        )
    )
    
    chunks = []
    for i, m in enumerate(cabeceras):
        ini = m.start()
        fin = cabeceras[i + 1].start() if i + 1 < len(cabeceras) else len(texto)
        parte = texto[ini:fin].strip()
        if parte:
            chunks.append(parte)
    return chunks

lista_textos_chunks_reclutamiento = chunkear_reclutamiento(texto_reclutamiento)

if len(lista_textos_chunks_reclutamiento) > 0:
    total_chunks = len(lista_textos_chunks_reclutamiento)
    tamano_promedio = sum(len(c) for c in lista_textos_chunks_reclutamiento) // total_chunks
    tamano_minimo = min(len(c) for c in lista_textos_chunks_reclutamiento)
    tamano_maximo = max(len(c) for c in lista_textos_chunks_reclutamiento)

    print("Documento utilizado:")
    print(ruta_archivo_reclutamiento)
    print(f"\nTotal de chunks: {total_chunks}")
    print(f"Tamaño promedio: {tamano_promedio} caracteres")
    print(f"Tamaño min/max: {tamano_minimo} / {tamano_maximo} caracteres\n")

    print("Inicio de cada chunk:")
    print("-" * 60)
    for i, c in enumerate(lista_textos_chunks_reclutamiento):
        primeros = c[:55].replace("\n", " ")
        print(f"  ch_{i:02d}: {primeros}...")
else:
    print("⚠️ ADVERTENCIA: No se detectaron los bloques principales.")

from langchain_core.documents import Document

chunks_reclutamiento = [
    Document(page_content=texto)
    for texto in lista_textos_chunks_reclutamiento
]

print("\nChunks de Reclutamiento y Onboarding convertidos correctamente a documentos LangChain.")

Documento utilizado:
C:\Users\USER\MI PROYECTO\Documentos\03_Reclutamiento_Onboarding.txt

Total de chunks: 4
Tamaño promedio: 299 caracteres
Tamaño min/max: 103 / 517 caracteres

Inicio de cada chunk:
------------------------------------------------------------
  ch_00: 1. PROCESO DE SELECCIÓN Requisición del área -> publica...
  ch_01: 2. PROGRAMA DE REFERIDOS - Cualquier colaborador puede ...
  ch_02: 3. ONBOARDING (INDUCCIÓN DE NUEVOS INGRESOS) Pasos del ...
  ch_03: 4. DOCUMENTOS DE INGRESO Identificación, datos bancario...

Chunks de Reclutamiento y Onboarding convertidos correctamente a documentos LangChain.


## 8.3. Generación de embeddings y creación del Vector Store

En esta sección se generan los embeddings del documento de Reclutamiento y Onboarding utilizando Google Gemini. Los embeddings permiten representar los fragmentos del documento como vectores para realizar búsquedas semánticas.

Estos vectores se almacenan en un Vector Store independiente de Chroma, que será utilizado posteriormente por el Agente RAG de Reclutamiento y Onboarding.


In [21]:
import os

# ============================================================
#   GENERACIÓN DE EMBEDDINGS Y CREACIÓN DEL VECTOR STORE
# ============================================================

# La ruta del Vector Store se obtiene desde la configuración
# central de rutas del proyecto: RUTA_VECTORSTORE_RECLUTAMIENTO

# 1. Crear la carpeta del Vector Store si no existe usando pathlib
RUTA_VECTORSTORE_RECLUTAMIENTO.mkdir(
    parents=True,
    exist_ok=True
)

# 2. Crear el modelo de embeddings de Google Gemini
embeddings = GoogleGenerativeAIEmbeddings(
    model=MODELO_EMBEDDING
)

# 3. Crear el Vector Store de Chroma
vectorstore_reclutamiento = Chroma.from_documents(
    documents=chunks_reclutamiento,
    embedding=embeddings,
    persist_directory=str(RUTA_VECTORSTORE_RECLUTAMIENTO)
)

# 4. Verificación del proceso
print("Vector Store de Reclutamiento y Onboarding creado correctamente.")
print("Ruta del Vector Store:")
print(RUTA_VECTORSTORE_RECLUTAMIENTO)

print("\nContenido generado por Chroma:")
for archivo in os.listdir(RUTA_VECTORSTORE_RECLUTAMIENTO):
    print("-", archivo)

Vector Store de Reclutamiento y Onboarding creado correctamente.
Ruta del Vector Store:
C:\Users\USER\MI PROYECTO\Vectorstores\reclutamiento_onboarding

Contenido generado por Chroma:
- 5ff8d196-ece9-4a1d-baf6-82c8927e0814
- chroma.sqlite3


## 8.4. Creación del Retriever del Agente de Reclutamiento y Onboarding

En esta sección se configura el retriever del Agente de Reclutamiento y Onboarding. El retriever realiza búsquedas semánticas dentro del Vector Store creado previamente y recupera los fragmentos más relevantes para cada consulta del usuario.

Estos fragmentos serán utilizados posteriormente por el modelo de lenguaje para generar respuestas basadas exclusivamente en la base documental de Reclutamiento y Onboarding.


In [22]:
# ============================================================
#  CREACIÓN DEL RETRIEVER DE RECLUTAMIENTO Y ONBOARDING
# ============================================================

# Convertimos el Vector Store de Reclutamiento y Onboarding en un Retriever.
# Esto nos permite realizar búsquedas semánticas dentro de esta base de conocimiento específica.
retriever_reclutamiento = vectorstore_reclutamiento.as_retriever(
    # Configuramos el parámetro k=5 para que el retriever devuelva 
    search_kwargs={"k": 5}
)

# Mostramos un mensaje en consola para verificar que el retriever se creó con éxito.
print("Retriever del Agente de Reclutamiento y Onboarding creado correctamente.")

Retriever del Agente de Reclutamiento y Onboarding creado correctamente.


## 8.5. Creación del Prompt del Agente de Reclutamiento y Onboarding

En esta sección se definen las instrucciones específicas del Agente de Reclutamiento y Onboarding. El prompt establece que el agente debe responder utilizando únicamente la información recuperada desde su propio Vector Store.

El agente no debe inventar información ni utilizar conocimientos externos. Si no encuentra información suficiente para responder una consulta, deberá indicarlo explícitamente.


In [23]:
# ============================================================
#  CREACIÓN DEL PROMPT DEL AGENTE DE RECLUTAMIENTO Y ONBOARDING
# ============================================================

# Definimos la plantilla de prompt para el Agente de Reclutamiento y Onboarding.
# Esta plantilla establece la identidad del agente y las reglas estrictas 
# para garantizar que solo responda utilizando su base documental asignada.
prompt_reclutamiento = """
Eres el Agente de Reclutamiento y Onboarding de PATITO S.A.

Tu función es responder preguntas relacionadas exclusivamente
con los procesos de reclutamiento, selección y onboarding de nuevos colaboradores en la empresa.

REGLAS IMPORTANTES:

1. Responde únicamente utilizando la información proporcionada
   en el contexto recuperado desde la base documental de
   Reclutamiento y Onboarding.

2. No utilices información de otras bases de conocimiento,
   agentes o documentos externos.

3. No inventes, supongas ni completes información que no esté
   presente en el contexto proporcionado.

4. Si la información necesaria para responder la pregunta
   no aparece en el contexto, responde exactamente:

"No encontré información suficiente en la base documental proporcionada"

5. Mantén las respuestas claras, directas y profesionales.

CONTEXTO DE RECLUTAMIENTO Y ONBOARDING:
{context}

PREGUNTA DEL USUARIO:
{question}

RESPUESTA:
"""

# Mostramos un mensaje de confirmación de que el prompt se creó exitosamente.
print("Prompt del Agente de Reclutamiento y Onboarding creado correctamente.")

Prompt del Agente de Reclutamiento y Onboarding creado correctamente.


## 8.6. Creación del Agente RAG de Reclutamiento y Onboarding

En esta sección se integran el modelo de lenguaje de Google Gemini, el Retriever y el prompt definido anteriormente para construir el Agente RAG de Reclutamiento y Onboarding.

El agente utilizará el Retriever para recuperar los fragmentos más relevantes del documento y proporcionarlos al modelo de lenguaje como contexto. De esta manera, generará respuestas basadas exclusivamente en la información disponible en su propia base documental.


In [24]:
# ============================================================
#  CREACIÓN DEL AGENTE DE RECLUTAMIENTO Y ONBOARDING
# ============================================================

# Definimos la función principal que encapsula la lógica del agente RAG.
# Recibe la pregunta del usuario y se encarga de todo el flujo de respuesta.
def agente_reclutamiento(pregunta):

    # 1. Utilizamos el retriever para buscar los fragmentos más relevantes
    # dentro del Vector Store de Reclutamiento y Onboarding basándonos en la pregunta.
    documentos_relevantes = retriever_reclutamiento.invoke(
        pregunta
    )

    # 2. Verificamos si el retriever encontró documentos o información suficiente.
    if not documentos_relevantes:
        return (
            "No encontré información suficiente en la "
            "base documental proporcionada."
        )

    # 3. Unimos el contenido de los documentos recuperados para formar un único contexto de texto.
    contexto = "\n\n".join(
        documento.page_content
        for documento in documentos_relevantes
    )

    # 4. Insertamos el contexto recuperado y la pregunta del usuario dentro de nuestra plantilla de prompt.
    prompt_final = prompt_reclutamiento.format(
        context=contexto,
        question=pregunta
    )

    # 5. Enviamos el prompt final al modelo de lenguaje (Gemini) manejando posibles errores de ejecución.
    try:
        respuesta = llm_beneficios.invoke(
            prompt_final
        )
        # Retornamos el texto con la respuesta generada por el modelo.
        return respuesta.content

    except Exception as e:
        # En caso de error, mostramos detalles técnicos en consola
        # y devolvemos un mensaje amigable al usuario.
        print("ERROR REAL DE GEMINI:")
        print(type(e).__name__)
        print(e)
        return (
            "No fue posible procesar la consulta "
            "debido a un error del servicio."
        )

# Mostramos un mensaje de confirmación de que la función del agente fue creada con éxito.
print("Agente de Reclutamiento y Onboarding creado correctamente.")

Agente de Reclutamiento y Onboarding creado correctamente.


## 9. Herramienta de Acción para Registro de Solicitudes

En esta sección se incorpora una herramienta de acción (Tool) que permitirá registrar y almacenar formalmente las solicitudes o trámites de los colaboradores.

A diferencia de los agentes RAG, que utilizan documentos para responder consultas, esta herramienta permitirá escribir y guardar las solicitudes de manera persistente en el archivo `registro_solicitudes_rrhh.txt`.


## 9.1. Creación de la herramienta de registro de solicitudes

En esta sección se crea una herramienta de acción que permite registrar las solicitudes de los colaboradores de PATITO S.A.

La herramienta recibe los datos del colaborador, el tipo de solicitud y el detalle de la petición, y almacena esta información de forma persistente en el archivo `registro_solicitudes_rrhh.txt`.

De esta manera, el sistema puede realizar una acción concreta además de consultar información mediante los agentes RAG.


In [25]:

import uuid
from datetime import datetime, timedelta
from pathlib import Path

# ============================================================
#   HERRAMIENTA DE ACCIÓN (REGISTRO DE SOLICITUDES)
# ============================================================

def registrar_solicitud(
    tipo_solicitud: str,
    colaborador: str = "N/A",
    detalle: str = "N/A", 
    fecha_inicio: str = "N/A",
    fecha_fin: str = "N/A", 
    dias: str = "N/A",
    jefe: str = "N/A", 
    nombre_dependiente: str = "N/A",
    vinculo: str = "N/A", 
    documentos_respaldo: str = "N/A"
) -> str:

    tipo = tipo_solicitud.strip().lower()
    
    # ============================================================
    # 1. VALIDACIÓN DE VACACIONES
    # ============================================================

    if "vacaciones" in tipo:

        if not all([
            colaborador != "N/A",
            fecha_inicio != "N/A",
            fecha_fin != "N/A",
            dias != "N/A",
            jefe != "N/A"
        ]):
            return (
                "Error: Faltan datos obligatorios para vacaciones "
                "(colaborador, fecha de inicio y fin, número de días "
                "y jefe que aprueba)."
            )
        
        # Validación de anticipación de 15 días
        try:
            f_inicio = datetime.strptime(
                fecha_inicio,
                "%Y-%m-%d"
            ).date()

            hoy = datetime.now().date()

            diferencia = (f_inicio - hoy).days
            
            if diferencia < 15:
                return (
                    f"Error: La solicitud de vacaciones debe realizarse "
                    f"con al menos 15 días de anticipación "
                    f"(faltan {diferencia} días)."
                )

        except ValueError:
            return (
                "Error: El formato de la fecha de inicio "
                "debe ser YYYY-MM-DD."
            )
            
        detalle_str = (
            f"Vacaciones del {fecha_inicio} al {fecha_fin} "
            f"({dias} días), aprobadas por el Ing. {jefe}."
        )

    # ============================================================
    # 2. VALIDACIÓN DE DEPENDIENTE
    # ============================================================

    elif "dependiente" in tipo:

        if not all([
            colaborador != "N/A",
            nombre_dependiente != "N/A",
            vinculo != "N/A",
            documentos_respaldo != "N/A"
        ]):
            return (
                "Error: Faltan datos obligatorios para la inscripción "
                "de dependiente (nombre del dependiente, vínculo "
                "y documentos de respaldo)."
            )
            
        detalle_str = (
            f"Inscripción de dependiente: {nombre_dependiente} "
            f"({vinculo}). Documentos de respaldo: "
            f"{documentos_respaldo}."
        )

    # ============================================================
    # 3. VALIDACIÓN DEL TIPO DE SOLICITUD
    # ============================================================

    else:
        return (
            "Error: Tipo de solicitud no reconocido. "
            "Solo se permite 'Vacaciones' o "
            "'Inscripción de dependiente'."
        )

    
# ============================================================
    # 4. SALVAGUARDA CONTRA DUPLICADOS 
 # ============================================================

    if Path(REGISTRO_PATH).exists():
        try:
            with open(str(REGISTRO_PATH), "r", encoding="utf-8") as archivo:
                lineas = archivo.readlines()

            for linea in lineas:
                linea_lower = linea.lower()
                # Verificamos si ya existe una solicitud activa para el mismo colaborador en las mismas fechas exactas
                if colaborador.strip().lower() in linea_lower and fecha_inicio in linea_lower and fecha_fin in linea_lower:
                    return f"Error: Ya existe una solicitud registrada para {colaborador} en las fechas del {fecha_inicio} al {fecha_fin}."

        except Exception as e:
            print(f"Aviso en validación de duplicados: {e}")

    # ============================================================
    # 5. REGISTRO DE LA SOLICITUD
    # ============================================================

    try:

        # Generar identificador único
        id_unico = str(uuid.uuid4())[:8].upper()

        # Obtener fecha y hora actual
        fecha_hora = datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        )
        
        # Guardar la solicitud
        with open(
            str(REGISTRO_PATH),
            "a",
            encoding="utf-8"
        ) as archivo:

            registro = (
                f"ID: {id_unico} | "
                f"Fecha: {fecha_hora} | "
                f"Colaborador: {colaborador} | "
                f"Tipo: {tipo_solicitud} | "
                f"Detalle: {detalle_str} | "
                f"Jefe: {jefe} | "
                f"Días: {dias}\n"
            )

            archivo.write(registro)
            
        return (
            f"Solicitud registrada exitosamente "
            f"con ID [{id_unico}] para el colaborador: "
            f"{colaborador}"
        )
        
    except Exception as e:

        print("ERROR AL ESCRIBIR EN EL REGISTRO:")
        print(type(e).__name__)
        print(e)

        return (
            "No fue posible registrar la solicitud "
            "debido a un error técnico."
        )


print(
    "Herramienta de acción "
    "(registro de solicitudes) creada correctamente."
)

print("Ruta del archivo de registro:")
print(REGISTRO_PATH)

Herramienta de acción (registro de solicitudes) creada correctamente.
Ruta del archivo de registro:
C:\Users\USER\MI PROYECTO\registro_solicitudes_rrhh.txt


## 10. Orquestador Central de Recursos Humanos

En esta sección se construye el orquestador central que coordinará los agentes y herramientas del sistema de Recursos Humanos mediante LangChain y Google Gemini.

El orquestador recibirá las consultas de los colaboradores y decidirá automáticamente qué agente o herramienta utilizar según el tema de la solicitud.

Además, contará con memoria de conversación mediante `InMemorySaver`, permitiendo mantener el contexto entre diferentes interacciones y completar solicitudes que requieran varios pasos.


## 10.1. Creación y configuración del Orquestador Central

En esta sección se crea y configura el Orquestador Central de Recursos Humanos utilizando LangChain y Google Gemini. El orquestador integra los tres agentes RAG y la herramienta de registro de solicitudes, permitiendo analizar cada consulta y dirigirla automáticamente al agente o herramienta correspondiente.

Además, se configura la memoria de conversación mediante `InMemorySaver` y las funciones necesarias para consultar el sistema, visualizar las herramientas utilizadas y obtener la respuesta final del asistente.


In [26]:
import uuid
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool

# ============================================================
#  EL ORQUESTADOR CENTRAL DE RECURSOS HUMANOS
# ============================================================

# ============================================================
# 1. DEFINICIÓN DE LAS TOOLS 
# ============================================================

@tool
def consultar_beneficios(pregunta: str) -> str:
    """
    Responde preguntas sobre beneficios, compensaciones,
    seguros médicos, bonos y otros beneficios corporativos.
    """
    return agente_beneficios(pregunta)


@tool
def consultar_reglamento(pregunta: str) -> str:
    """
    Responde preguntas sobre el Reglamento Interno,
    jornada laboral, vacaciones, permisos, código de conducta
    y faltas y sanciones.
    """
    return agente_reglamento(pregunta)


@tool
def consultar_reclutamiento(pregunta: str) -> str:
    """
    Responde preguntas sobre procesos de selección,
    vacantes, programa de referidos e inducción (onboarding).
    """
    return agente_reclutamiento(pregunta)


@tool
def registrar_solicitud_rrhh(
    tipo_solicitud: str,
    colaborador: str = "N/A",
    detalle: str = "N/A",
    fecha_inicio: str = "N/A",
    fecha_fin: str = "N/A",
    dias: str = "N/A",
    jefe: str = "N/A",
    nombre_dependiente: str = "N/A",
    vinculo: str = "N/A",
    documentos_respaldo: str = "N/A"
) -> str:
    """
    Registra formalmente una solicitud o trámite de Recursos Humanos (Vacaciones o Inscripción de dependiente)
    en un archivo de texto plano tras validar todos los datos obligatorios y obtener confirmación.
    """
    return registrar_solicitud(
        tipo_solicitud=tipo_solicitud,
        colaborador=colaborador,
        detalle=detalle,
        fecha_inicio=fecha_inicio,
        fecha_fin=fecha_fin,
        dias=dias,
        jefe=jefe,
        nombre_dependiente=nombre_dependiente,
        vinculo=vinculo,
        documentos_respaldo=documentos_respaldo
    )


# ============================================================
# 2. AGRUPAR TODAS LAS TOOLS DEL ORQUESTADOR
# ============================================================

tools_orquestador = [
    consultar_beneficios,
    consultar_reglamento,
    consultar_reclutamiento,
    registrar_solicitud_rrhh
]


# ============================================================
# 3. SYSTEM PROMPT DEL ORQUESTADOR (CON EL SISTEMA DE CONTROL)
# ============================================================

SYSTEM_PROMPT = """
Eres el orquestador central de la Mesa de Ayuda de Recursos Humanos de PATITO S.A.

Tu función es analizar las preguntas de los colaboradores y utilizar la herramienta especializada correspondiente.

SISTEMA DE CONTROL ESTRICTO Y OBLIGATORIO:
Antes de invocar bajo ninguna circunstancia la herramienta 'registrar_solicitud_rrhh', el agente DEBE verificar que posea TODOS los siguientes datos obligatorios:

1. **Para Vacaciones (OBLIGATORIOS):**
   - Nombre del colaborador.
   - Fecha de inicio y fecha de fin.
   - Número de días exactos.
   - Jefe que aprueba.
   - Validación de anticipación (al menos 15 días respecto a la fecha actual).

2. **Para Inscripción de Dependiente (OBLIGATORIOS):**
   - Nombre del dependiente.
   - Vínculo familiar.
   - Documentos de respaldo.

REGLAS DE BLOQUEO Y FLUJO:
- **PROHIBIDO REGISTRAR SI FALTA UN SOLO DATO:** Si el usuario te da una frase incompleta (por ejemplo, solo su nombre y fechas), **NO LLAMES A LA HERRAMIENTA**. Debes responderle exclusivamente pidiéndole los datos faltantes (número de días y jefe que aprueba).
- **CONFIRMACIÓN PREVIA:** Una vez que tengas absolutamente todos los datos completos y validados, preséntale un resumen al usuario y **pídele confirmación explícita**. Solo cuando el usuario responda "Sí, confirmo", tienes permiso para ejecutar la herramienta de registro.
"""

# ============================================================
# 4. CREACIÓN DE LA MEMORIA
# ============================================================

memoria = InMemorySaver()


# ============================================================
# 5. CREACIÓN DEL ORQUESTADOR
# ============================================================

orquestador = create_agent(
    model=llm,
    tools=tools_orquestador,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=memoria,
)


# ============================================================
# 6. VERIFICACIÓN DEL ORQUESTADOR
# ============================================================

print(
    "Orquestador de Recursos Humanos creado "
    "exitosamente con las siguientes tools:"
)

for t in tools_orquestador:
    print(f"  - {t.name}")


# ============================================================
# 7. FUNCIÓN PARA MOSTRAR LAS TOOLS UTILIZADAS
# ============================================================

def _imprimir_pasos(resultado):

    for m in resultado["messages"]:

        # Muestra las herramientas llamadas por el orquestador
        for tc in (getattr(m, "tool_calls", None) or []):
            print(
                f"[TOOL USADA] "
                f"{tc['name']}({tc['args']})"
            )

        # Muestra la respuesta obtenida de cada herramienta
        if m.__class__.__name__ == "ToolMessage":
            print(
                f"[RESPUESTA DE TOOL] "
                f"{str(m.content)[:300]}\n"
            )


# ============================================================
# 8. FUNCIÓN PARA EXTRAER EL TEXTO DE LA RESPUESTA
# ============================================================

def extraer_texto(content):

    # Si la respuesta ya es texto
    if isinstance(content, str):
        return content

    # Si la respuesta viene como una lista de bloques
    if isinstance(content, list):

        partes = []

        for b in content:

            if isinstance(b, dict):
                partes.append(
                    b.get("text", "")
                )

            elif isinstance(b, str):
                partes.append(b)

        return "".join(partes).strip()

    # Para cualquier otro tipo de contenido
    return str(content)


# ============================================================
# 9. FUNCIÓN PRINCIPAL PARA CONSULTAR AL ORQUESTADOR
# ============================================================

def consultar_rrhh(
    pregunta: str,
    thread_id: str = None
):
    thread_id = (
        thread_id
        or f"rrhh-{uuid.uuid4().hex[:8]}"
    )

    config = {
        "configurable": {
            "thread_id": thread_id
        }
    }

    print(f">>> Colaborador: {pregunta}\n")

    resultado = orquestador.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": pregunta
                }
            ]
        },
        config
    )

    herramientas_usadas = []

    for mensaje in resultado["messages"]:
        for tool_call in (
            getattr(mensaje, "tool_calls", None) or []
        ):
            nombre_tool = tool_call["name"]

            if nombre_tool not in herramientas_usadas:
                herramientas_usadas.append(nombre_tool)

    print("Herramientas utilizadas:")

    for herramienta in herramientas_usadas:
        print(f"- {herramienta}")

    respuesta_final = extraer_texto(
        resultado["messages"][-1].content
    )

    print("\n=== Respuesta final del Asistente ===")
    print(respuesta_final)

Orquestador de Recursos Humanos creado exitosamente con las siguientes tools:
  - consultar_beneficios
  - consultar_reglamento
  - consultar_reclutamiento
  - registrar_solicitud_rrhh


-----------------------------------------------------------------------------------------
-----------------------------------------------------------------------------------------

Prueba

In [27]:
consultar_rrhh(
    "Voy a tomar mis vacaciones y además quiero agregar a mi pareja al seguro médico. "
    "¿Cuántos días me corresponden, cómo los solicito y qué necesito para inscribir a un dependiente en el beneficio?"
)

>>> Colaborador: Voy a tomar mis vacaciones y además quiero agregar a mi pareja al seguro médico. ¿Cuántos días me corresponden, cómo los solicito y qué necesito para inscribir a un dependiente en el beneficio?

Herramientas utilizadas:
- consultar_reglamento
- consultar_beneficios

=== Respuesta final del Asistente ===
Para responder a tus consultas, aquí tienes la información necesaria:

### Sobre tus Vacaciones:
Tienes derecho a **15 días hábiles** por año cumplido. Para solicitarlas, debes cumplir con lo siguiente:
*   Realizar la solicitud con al menos **15 días de anticipación**.
*   Contar con la **aprobación de tu jefe directo**.

Si deseas que proceda con el registro de tu solicitud, por favor facilítame los siguientes datos:
1. Nombre completo del colaborador.
2. Fecha de inicio y fecha de fin de las vacaciones.
3. Número exacto de días.
4. Nombre de tu jefe directo que aprueba la solicitud.

### Sobre la Inscripción de tu Pareja:
Para inscribir a tu pareja en el seguro médic

## 11. Mini App — Mesa de Ayuda de Recursos Humanos

Se desarrolla una mini aplicación que permite al usuario interactuar con la Mesa de Ayuda de Recursos Humanos mediante lenguaje natural.

La aplicación utiliza el **orquestador central** para analizar cada consulta y dirigirla al agente especializado correspondiente. De esta manera, el usuario puede realizar consultas sobre Beneficios y Compensaciones, Reglamento Interno y Reclutamiento y Onboarding, así como gestionar solicitudes de Recursos Humanos.

El flujo de la aplicación es:

**Usuario → Mini App → Orquestador → Agente especializado → Respuesta final**

Cuando el usuario solicita registrar una gestión de RR. HH., el orquestador utiliza el **agente de acción** y la herramienta `registrar_solicitud()`, que valida los datos requeridos antes de realizar el registro en `registro_solicitudes_rrhh.txt`.

La mini app permite comprobar el funcionamiento integrado de los agentes RAG, el agente de acción y el orquestador en una única interfaz.

In [34]:
import ipywidgets as widgets
import html as html_lib
import re
import uuid
from datetime import datetime
from pathlib import Path
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool

# ============================================================
# CONFIGURACIÓN CENTRAL DE RUTAS DEL PROYECTO
# ============================================================
RUTA_PROYECTO = Path.cwd() 

RUTA_DOCUMENTOS = RUTA_PROYECTO / "Documentos"
RUTA_DOCUMENTO_BENEFICIOS = RUTA_DOCUMENTOS / "01_Beneficios_Compensaciones.txt"
RUTA_DOCUMENTO_REGLAMENTO = RUTA_DOCUMENTOS / "02_Reglamento_Interno.txt"
RUTA_DOCUMENTO_RECLUTAMIENTO = RUTA_DOCUMENTOS / "03_Reclutamiento_Onboarding.txt"

RUTA_VECTORSTORES = RUTA_PROYECTO / "Vectorstores"
RUTA_VECTORSTORE_BENEFICIOS = RUTA_VECTORSTORES / "beneficios_compensaciones"
RUTA_VECTORSTORE_REGLAMENTO = RUTA_VECTORSTORES / "reglamento_interno"
RUTA_VECTORSTORE_RECLUTAMIENTO = RUTA_VECTORSTORES / "reclutamiento_onboarding"

REGISTRO_PATH = RUTA_PROYECTO / "registro_solicitudes_rrhh.txt"

print("Ruta principal del proyecto:", RUTA_PROYECTO)
print("Archivo de registro vinculado:", REGISTRO_PATH.exists())


def extraer_texto(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        partes = []
        for b in content:
            if isinstance(b, dict):
                partes.append(b.get("text", ""))
            elif isinstance(b, str):
                partes.append(b)
        return "".join(partes).strip()
    return str(content)


# ============================================================
# HERRAMIENTA DE REGISTRO VALIDADA (Vacaciones y Dependientes)
# ============================================================
def registrar_solicitud(tipo_solicitud: str, colaborador: str = "N/A", detalle: str = "N/A", 
                        fecha_inicio: str = "N/A", fecha_fin: str = "N/A", 
                        dias: str = "N/A", jefe: str = "N/A", 
                        nombre_dependiente: str = "N/A", vinculo: str = "N/A", 
                        documentos_respaldo: str = "N/A") -> str:
    tipo = tipo_solicitud.strip().lower()
    
    if "vacaciones" in tipo:
        if not all([colaborador != "N/A", fecha_inicio != "N/A", fecha_fin != "N/A", dias != "N/A", jefe != "N/A"]):
            return "Error: Faltan datos obligatorios para vacaciones (colaborador, fecha de inicio y fin, número de días y jefe que aprueba)."
        
        try:
            f_inicio = datetime.strptime(fecha_inicio, "%Y-%m-%d")
            hoy = datetime.now()
            diferencia = (f_inicio - hoy).days
            if diferencia < 15:
                return f"Error: La solicitud de vacaciones debe realizarse con al menos 15 días de anticipación (faltan {diferencia} días)."
        except ValueError:
            return "Error: El formato de la fecha de inicio debe ser YYYY-MM-DD."
            
        detalle_str = f"Vacaciones del {fecha_inicio} al {fecha_fin} ({dias} días), aprobadas por el Ing. {jefe}."
        
    elif "dependiente" in tipo:
        if not all([colaborador != "N/A", nombre_dependiente != "N/A", vinculo != "N/A", documentos_respaldo != "N/A"]):
            return "Error: Faltan datos obligatorios para la inscripción de dependiente (colaborador, nombre del dependiente, vínculo y documentos de respaldo)."
            
        detalle_str = f"Inscripción de dependiente: {nombre_dependiente} ({vinculo}). Documentos de respaldo: {documentos_respaldo}."
    else:
        return "Error: Tipo de solicitud no reconocido. Solo se permite 'Vacaciones' o 'Inscripción de dependiente'."

    # Salvaguarda contra duplicados
    if REGISTRO_PATH.exists():
        try:
            with open(str(REGISTRO_PATH), "r", encoding="utf-8") as archivo:
                lineas = archivo.readlines()
            for linea in lineas:
                linea_lower = linea.lower()
                if colaborador.strip().lower() in linea_lower and fecha_inicio in linea_lower and fecha_fin in linea_lower:
                    return f"Error: Ya existe una solicitud registrada para {colaborador} en las fechas del {fecha_inicio} al {fecha_fin}."
        except Exception as e:
            print(f"Aviso en validación de duplicados: {e}")

    try:
        id_unico = str(uuid.uuid4())[:8].upper()
        fecha_hora = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        with open(str(REGISTRO_PATH), "a", encoding="utf-8") as archivo:
            registro = f"ID: {id_unico} | Fecha: {fecha_hora} | Colaborador: {colaborador} | Tipo: {tipo_solicitud} | Detalle: {detalle_str} | Jefe: {jefe} | Días: {dias}\n"
            archivo.write(registro)
            
        return f"Solicitud registrada exitosamente con ID [{id_unico}] para el colaborador: {colaborador}"
        
    except Exception as e:
        return "No fue posible registrar la solicitud debido a un error técnico."


# ============================================================
# DEFINICIÓN DE TOOLS DEL ORQUESTADOR
# ============================================================
@tool
def consultar_beneficios(pregunta: str) -> str:
    """Responde preguntas sobre beneficios, compensaciones y seguros médicos."""
    return agente_beneficios(pregunta)

@tool
def consultar_reglamento(pregunta: str) -> str:
    """Responde preguntas sobre el Reglamento Interno, vacaciones y permisos."""
    return agente_reglamento(pregunta)

@tool
def consultar_reclutamiento(pregunta: str) -> str:
    """Responde preguntas sobre procesos de selección, vacantes y onboarding."""
    return agente_reclutamiento(pregunta)

@tool
def registrar_solicitud_rrhh(
    tipo_solicitud: str,
    colaborador: str = "N/A",
    detalle: str = "N/A",
    fecha_inicio: str = "N/A",
    fecha_fin: str = "N/A",
    dias: str = "N/A",
    jefe: str = "N/A",
    nombre_dependiente: str = "N/A",
    vinculo: str = "N/A",
    documentos_respaldo: str = "N/A"
) -> str:
    """Registra formalmente una solicitud de Recursos Humanos (Vacaciones o Inscripción de dependiente)."""
    return registrar_solicitud(
        tipo_solicitud=tipo_solicitud, colaborador=colaborador, detalle=detalle,
        fecha_inicio=fecha_inicio, fecha_fin=fecha_fin, dias=dias, jefe=jefe,
        nombre_dependiente=nombre_dependiente, vinculo=vinculo, documentos_respaldo=documentos_respaldo
    )

tools_orquestador = [
    consultar_beneficios,
    consultar_reglamento,
    consultar_reclutamiento,
    registrar_solicitud_rrhh
]


# ============================================================
# SYSTEM PROMPT ACTUALIZADO CON CONTROLES ESTRICTOS
# ============================================================
from datetime import datetime

fecha_actual_sistema = datetime.now().strftime("%Y-%m-%d")

SYSTEM_PROMPT = f"""
Eres el orquestador central de la Mesa de Ayuda de Recursos Humanos de PATITO S.A.
La fecha actual del sistema es: {fecha_actual_sistema}. Usa obligatoriamente esta fecha para calcular la anticipación de las vacaciones.

Tu función es analizar las preguntas de los colaboradores y utilizar la herramienta especializada correspondiente.

SISTEMA DE CONTROL ESTRICTO Y OBLIGATORIO:
Antes de invocar bajo ninguna circunstancia la herramienta 'registrar_solicitud_rrhh', el agente DEBE verificar que posea TODOS los datos obligatorios según el tipo de solicitud:

1. **Para Vacaciones (OBLIGATORIOS):**
   - Nombre del colaborador (`colaborador`).
   - Fecha de inicio (`fecha_inicio` en formato YYYY-MM-DD) y fecha de fin (`fecha_fin`).
   - Número de días exactos (`dias`).
   - Jefe que aprueba (`jefe`).
   - Validación de anticipación: La fecha de inicio debe ser con al menos 15 días de anticipación respecto a la fecha actual del sistema ({fecha_actual_sistema}).

2. **Para Inscripción de Dependiente (OBLIGATORIOS):**
   - Nombre del colaborador (`colaborador`).
   - Nombre del dependiente (`nombre_dependiente`).
   - Vínculo familiar (`vinculo`).
   - Documentos de respaldo (`documentos_respaldo`).

REGLAS DE BLOQUEO Y FLUJO:
- **PROHIBIDO REGISTRAR SI FALTA UN SOLO DATO:** Si el usuario te da una frase incompleta, **NO LLAMES A LA HERRAMIENTA**. Debes responderle exclusivamente pidiéndole los datos faltantes.
- **CONFIRMACIÓN PREVIA:** Una vez que tengas absolutamente todos los datos completos y validados, preséntale un resumen al usuario y **pídele confirmación explícita**. Solo cuando el usuario responda "Sí, confirmo", tienes permiso para ejecutar la herramienta de registro pasando cada parámetro correspondiente.
"""


# ============================================================
# 1. Agente de chat con memoria (checkpointer + thread_id)
# ============================================================
memoria_chat = InMemorySaver()
agente_chat = create_agent(
    model=llm,
    tools=tools_orquestador,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=memoria_chat,
)
config_chat = {"configurable": {"thread_id": "chat-rrhh"}}
historial_chat = []


# ============================================================
# 2. CSS unificado (Cambiado a tonos azulitos corporativos)
# ============================================================
GRADIENT = "linear-gradient(135deg, #2b5876 0%, #4e4376 100%)"
CSS_APP = f"""
<style>
.sml-app * {{ box-sizing: border-box; }}
.sml-card {{
    background: white; border-radius: 14px; padding: 16px;
    box-shadow: 0 2px 10px rgba(0,0,0,0.06);
    font-family: -apple-system, 'Segoe UI', system-ui, sans-serif;
}}
.sml-header {{
    display: flex; align-items: center; gap: 8px;
    color: white; background: {GRADIENT};
    padding: 12px 16px; margin: -16px -16px 14px -16px;
    border-radius: 14px 14px 0 0;
    font-weight: 600; font-size: 1.05em;
}}
.sml-search-wrap {{ max-height: 540px; overflow-y: auto; }}
.sml-table {{ width: 100%; border-collapse: collapse; font-size: 0.86em; background: white; }}
.sml-table thead th {{
    background: #f0f4f8; color: #444; text-align: left;
    text-transform: uppercase; font-size: 0.72em; letter-spacing: 0.5px;
    padding: 10px 8px; border-bottom: 2px solid #d9e2ec;
    position: sticky; top: 0; z-index: 1;
}}
.sml-table tbody tr {{ border-bottom: 1px solid #f0f4f8; transition: background 0.12s ease; }}
.sml-table tbody tr:hover {{ background: #f0f4f8; }}
.sml-table td {{ padding: 9px 8px; vertical-align: middle; }}
.sml-table .col-id {{ font-weight: 700; color: #2b5876; }}
.sml-table .col-tipo {{ font-weight: 600; color: #3182ce; }}
.sml-vacio {{ text-align: center; padding: 40px 20px; color: #9aa5b1; }}
.sml-vacio .icon {{ font-size: 2.5em; display: block; margin-bottom: 8px; opacity: 0.5; }}
.sml-resumen {{
    color: #555; font-size: 0.82em; margin-bottom: 10px;
    padding: 6px 10px; background: #f0f4f8; border-radius: 6px;
}}
.sml-chat-box {{
    height: 460px; overflow-y: auto; padding: 14px; border-radius: 10px;
    background: linear-gradient(180deg, #f7fafc 0%, #edf2f7 100%);
    border: 1px solid #d9e2ec; display: flex; flex-direction: column-reverse;
}}
.sml-msg-row {{ display: flex; margin: 12px 0; animation: sml-in 0.25s ease-out; }}
@keyframes sml-in {{ from {{ opacity: 0; transform: translateY(6px); }} to {{ opacity: 1; transform: translateY(0); }} }}
.sml-msg-row.user  {{ justify-content: flex-end; }}
.sml-msg-row.agent {{ justify-content: flex-start; }}
.sml-bubble {{ max-width: 82%; padding: 11px 15px; border-radius: 16px; line-height: 1.5; word-wrap: break-word; font-size: 0.93em; }}
.sml-bubble.user {{ background: {GRADIENT}; color: white; border-bottom-right-radius: 4px; }}
.sml-bubble.agent {{ background: white; color: #2d3748; border: 1px solid #d9e2ec; border-bottom-left-radius: 4px; box-shadow: 0 1px 3px rgba(0,0,0,0.04); }}
.sml-sender {{ font-size: 0.72em; font-weight: 700; opacity: 0.75; margin-bottom: 5px; letter-spacing: 0.3px; }}
.sml-trazas {{ background: #ebf8ff; border-left: 3px solid #3182ce; padding: 9px 12px; margin-bottom: 8px; border-radius: 6px; font-family: 'SF Mono', Consolas, monospace; font-size: 0.78em; color: #4a5568; }}
.sml-trazas-titulo {{ display: flex; align-items: center; gap: 5px; font-weight: 700; color: #2b6cb0; margin-bottom: 4px; font-size: 0.85em; }}
.sml-trazas .accion {{ color: #2b6cb0; }}
.sml-trazas .obs {{ color: #2f855a; }}
.sml-chat-vacio {{ text-align: center; padding: 80px 20px; color: #9aa5b1; margin: auto; }}
.sml-chat-vacio .icon {{ font-size: 3em; display: block; margin-bottom: 10px; opacity: 0.4; }}
.sml-btn-gradient button, .sml-btn-gradient {{
    background: {GRADIENT} !important; color: white !important; border: none !important;
    font-weight: 600 !important; box-shadow: 0 2px 6px rgba(43,88,118,0.25) !important;
    transition: transform 0.1s ease, box-shadow 0.15s ease !important;
}}
.sml-btn-gradient button:hover, .sml-btn-gradient:hover {{ transform: translateY(-1px) !important; box-shadow: 0 4px 10px rgba(43,88,118,0.35) !important; }}
.sml-btn-gradient-outline button, .sml-btn-gradient-outline {{ background: white !important; color: #2b5876 !important; border: 2px solid #2b5876 !important; font-weight: 600 !important; }}
.sml-btn-gradient-outline button:hover, .sml-btn-gradient-outline:hover {{ background: #f0f4f8 !important; }}
.sml-input-gradient {{ background: {GRADIENT}; padding: 2px; border-radius: 10px; box-shadow: 0 2px 8px rgba(43,88,118,0.15); }}
.sml-input-gradient textarea {{ border: none !important; border-radius: 8px !important; background: white !important; padding: 8px 12px !important; font-family: inherit !important; font-size: 0.92em !important; outline: none !important; }}
</style>
"""


# ============================================================
# 3. Panel izquierdo: registro de solicitudes RRHH (sin botón de eliminar ID)
# ============================================================
def _leer_registros(filtro=""):
    if not Path(REGISTRO_PATH).exists():
        return []
    filas = []
    for linea in Path(REGISTRO_PATH).read_text(encoding="utf-8").splitlines():
        if not linea.strip():
            continue
        reg = {"colaborador": "", "tipo_solicitud": "", "detalle": "", "fecha": "Reciente"}
        if "Colaborador:" in linea:
            try:
                partes = [p.strip() for p in linea.split("|")]
                datos_extra = []
                for p in partes:
                    if p.startswith("ID:"):
                        reg["id"] = p.replace("ID:", "").strip()
                    elif p.startswith("Colaborador:"):
                        reg["colaborador"] = p.replace("Colaborador:", "").strip()
                    elif p.startswith("Tipo:"):
                        reg["tipo_solicitud"] = p.replace("Tipo:", "").strip()
                    elif p.startswith("Detalle:"):
                        reg["detalle"] = p.replace("Detalle:", "").strip()
                    elif p.startswith("Jefe:"):
                        jefe = p.replace("Jefe:", "").strip()
                        if jefe and jefe.upper() != "N/A":
                            datos_extra.append(f"Jefe: {jefe}")
                    elif p.startswith("Días:"):
                        dias = p.replace("Días:", "").strip()
                        if dias and dias.upper() != "N/A":
                            datos_extra.append(f"Días: {dias}")
                    elif p.startswith("Fecha:"):
                        reg["fecha"] = p.replace("Fecha:", "").strip()
                
                if reg["fecha"] == "Reciente" or not reg["fecha"]:
                    for p in partes:
                        if "-" in p and ":" in p and len(p) >= 15:
                            reg["fecha"] = p.replace("Fecha:", "").strip()
                
                extra_str = f" ({', '.join(datos_extra)})" if datos_extra else ""
                id_str = f" [ID: {reg.get('id', '')}]" if reg.get('id') else ""
                reg["detalle_completo"] = f"{reg.get('detalle', '')}{extra_str}{id_str}"
            except Exception:
                reg["detalle_completo"] = linea
        else:
            partes = [x.strip() for x in linea.split("|")]
            reg["colaborador"] = partes[0] if len(partes) > 0 else "Desconocido"
            reg["tipo_solicitud"] = partes[1] if len(partes) > 1 else "General"
            reg["detalle_completo"] = partes[2] if len(partes) > 2 else linea
            reg["fecha"] = partes[3] if len(partes) > 3 else "Reciente"

        texto_busqueda = f"{reg['colaborador']} {reg['tipo_solicitud']} {reg['detalle_completo']} {reg['fecha']}".lower()
        if not filtro.strip() or filtro.strip().lower() in texto_busqueda:
            filas.append({
                "colaborador": reg["colaborador"],
                "tipo_solicitud": reg["tipo_solicitud"],
                "detalle": reg["detalle_completo"],
                "fecha": reg["fecha"]
            })
    return filas


def _render_tabla(filtro=""):
    registros = _leer_registros(filtro)
    if not registros:
        return ('<div class="sml-card sml-app">'
                '<div class="sml-header">📋 Solicitudes de RR. HH. registradas</div>'
                '<div class="sml-vacio"><span class="icon">📂</span>No hay solicitudes registradas aún.</div>'
                '</div>')
    
    html = ['<div class="sml-card sml-app">',
            '<div class="sml-header">📋 Solicitudes de RR. HH. registradas</div>',
            f'<div class="sml-resumen">Mostrando {len(registros)} solicitud(es)</div>',
            '<div class="sml-search-wrap">',
            '<table class="sml-table"><thead><tr>',
            '<th>Colaborador</th><th>Tipo</th><th>Detalle</th><th>Fecha</th>',
            '</tr></thead><tbody>']
    
    for r in reversed(registros):
        html.append(f'<tr>'
                    f'<td style="font-weight:600;">{html_lib.escape(r["colaborador"])}</td>'
                    f'<td class="col-tipo">{html_lib.escape(r["tipo_solicitud"])}</td>'
                    f'<td style="color:#444;">{html_lib.escape(r["detalle"])}</td>'
                    f'<td style="white-space:nowrap;color:#666;font-size:0.9em;">{html_lib.escape(r["fecha"])}</td>'
                    f'</tr>')
    html.append('</tbody></table></div></div>')
    return "".join(html)

input_buscar = widgets.Text(placeholder="Filtrar por colaborador/tipo...", layout=widgets.Layout(width="65%"))
btn_buscar = widgets.Button(description="Buscar", icon="search")
btn_ver_todas = widgets.Button(description="Ver todas", icon="list")

tabla_widget = widgets.HTML(value=_render_tabla())

def hacer_busqueda(_=None):
    tabla_widget.value = _render_tabla(input_buscar.value)

def ver_todas(_=None):
    input_buscar.value = ""
    tabla_widget.value = _render_tabla()

btn_buscar.on_click(hacer_busqueda)
btn_ver_todas.on_click(ver_todas)

panel_busqueda = widgets.VBox([
    widgets.HBox([input_buscar, btn_buscar, btn_ver_todas]),
    tabla_widget
], layout=widgets.Layout(width="46%", padding="0 8px 0 0"))


# ============================================================
# 4. Panel derecho: chat multi-turno con memoria + trazas
# ============================================================
def _render_chat():
    if not historial_chat:
        return ('<div class="sml-chat-box"><div class="sml-chat-vacio">'
                '<span class="icon">💬</span>Escribe una consulta o solicitud de RR. HH. para empezar.</div></div>')
    partes = ['<div class="sml-chat-box">']
    for msg in reversed(historial_chat):
        if msg["rol"] == "user":
            partes.append(
                f'<div class="sml-msg-row user"><div class="sml-bubble user">'
                f'<div class="sml-sender">👤 TU</div>{html_lib.escape(msg["contenido"])}'
                f'</div></div>'
            )
        else:
            partes.append('<div class="sml-msg-row agent"><div class="sml-bubble agent">')
            partes.append('<div class="sml-sender">🤖 ASISTENTE RR. HH.</div>')
            if msg.get("trazas"):
                partes.append('<div class="sml-trazas">')
                partes.append('<div class="sml-trazas-titulo">⚙️ Herramientas activadas</div>')
                for paso in msg["trazas"]:
                    if paso["tipo"] == "accion":
                        partes.append(
                            f'<div class="accion">→ {html_lib.escape(paso["tool"])}({html_lib.escape(str(paso["args"]))})</div>'
                        )
                    else:
                        partes.append(
                            f'<div class="obs">✓ {html_lib.escape(paso["tool"])} → {html_lib.escape(paso["preview"])}</div>'
                        )
                partes.append('</div>')
            partes.append(f'<div>{_md_a_html(msg["contenido"])}</div></div></div>')
    partes.append('</div>')
    return "".join(partes)

def _md_a_html(texto):
    safe = html_lib.escape(texto)
    safe = re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", safe)
    safe = re.sub(r"`([^`]+)`", r"<code>\1</code>", safe)
    return safe.replace("\n", "<br>")

chat_box = widgets.HTML(value=_render_chat())
input_chat = widgets.Textarea(placeholder="Escribe tu consulta o solicitud a Recursos Humanos...",
                            layout=widgets.Layout(width="100%", height="60px"))
input_chat.add_class("sml-input-gradient")
btn_enviar = widgets.Button(description="Enviar", icon="paper-plane", layout=widgets.Layout(width="auto"))
btn_reiniciar = widgets.Button(description="Limpiar", icon="trash", layout=widgets.Layout(width="auto"))
btn_enviar.add_class("sml-btn-gradient")
btn_reiniciar.add_class("sml-btn-gradient-outline")
estado_chat = widgets.HTML(value="")

def _enviar(_=None):
    pregunta = input_chat.value.strip()
    if not pregunta:
        return
    btn_enviar.disabled = True
    estado_chat.value = '<i style="color:#666;font-size:0.85em;">⏳ El asistente está consultando las bases de datos...</i>'
    input_chat.value = ""
    historial_chat.append({"rol": "user", "contenido": pregunta})
    chat_box.value = _render_chat()
    try:
        resultado = agente_chat.invoke(
            {"messages": [{"role": "user", "content": pregunta}]},
            config=config_chat,
        )
        msgs = resultado["messages"]
        idx = max((i for i, m in enumerate(msgs) if type(m).__name__ == "HumanMessage"), default=-1)
        trazas, respuesta_final = [], ""
        for m in msgs[idx + 1:]:
            tipo = type(m).__name__
            if tipo == "AIMessage" and getattr(m, "tool_calls", None):
                for tc in m.tool_calls:
                    trazas.append({"tipo": "accion", "tool": tc["name"], "args": tc["args"]})
            elif tipo == "ToolMessage":
                contenido_tool = m.content
                if isinstance(contenido_tool, list):
                    texto_plano = "".join([b.get("text", "") if isinstance(b, dict) else str(b) for b in contenido_tool])
                else:
                    texto_plano = str(contenido_tool or "")
                preview = texto_plano.replace("\n", " ")[:250]
                trazas.append({"tipo": "obs", "tool": m.name, "preview": preview})    
            elif tipo == "AIMessage" and m.content and not getattr(m, "tool_calls", None):
                respuesta_final = extraer_texto(m.content)
        if not respuesta_final:
            respuesta_final = "_(El asistente no generó una respuesta final para este turno.)_"
        historial_chat.append({"rol": "agent", "contenido": respuesta_final, "trazas": trazas})
        chat_box.value = _render_chat()
        tabla_widget.value = _render_tabla(input_buscar.value)
    except Exception as e:
        import traceback
        historial_chat.append({"rol": "agent", "contenido": f"⚠️ **Error:** {type(e).__name__}: {e}", "trazas": []})
        chat_box.value = _render_chat()
        traceback.print_exc()
    finally:
        estado_chat.value = ""
        btn_enviar.disabled = False

def _reiniciar(_=None):
    global config_chat
    historial_chat.clear()
    config_chat = {"configurable": {"thread_id": f"chat-rrhh-{id(historial_chat)}"}}
    chat_box.value = _render_chat()

btn_enviar.on_click(_enviar)
btn_reiniciar.on_click(_reiniciar)

panel_chat = widgets.VBox([
    widgets.HTML('<div class="sml-card sml-app"><div class="sml-header">💬 Asistente Virtual de RR. HH.</div>'),
    chat_box,
    estado_chat,
    input_chat,
    widgets.HBox([btn_enviar, btn_reiniciar]),
    widgets.HTML('</div>'),
], layout=widgets.Layout(width="54%", padding="0 0 0 8px"))


# ============================================================
# 5. Composición final: CSS + los dos paneles lado a lado
# ============================================================
display(widgets.HTML(CSS_APP))
display(widgets.HBox([panel_busqueda, panel_chat],
                     layout=widgets.Layout(width="100%", align_items="flex-start")))

Ruta principal del proyecto: C:\Users\USER\MI PROYECTO
Archivo de registro vinculado: False


HTML(value="\n<style>\n.sml-app * { box-sizing: border-box; }\n.sml-card {\n    background: white; border-radi…